In [1]:
!pip install transformers x-transformers sentencepiece --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00


In [2]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install tokenizers==0.20.3 transformers==4.46.3

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 75.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 51.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 35.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.

In [3]:
from sklearn.metrics import accuracy_score

In [4]:
import re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from transformers import AutoTokenizer, AutoModel
from x_transformers import Decoder


In [5]:
BATCH_SIZE = 64
LR = 2e-4
EPOCHS = 100
MAX_LEN = 256

DECODER_DIM = 768
DECODER_DEPTH = 2
DECODER_HEADS = 8

labels = [
    "no_depression",
    "depression"
]

NUM_CLASSES = len(labels)


In [6]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
import json

TRAIN_PATH = "/kaggle/input/datasets/kmkimmy/thai-depression/Thai Depression/train.json"
VALID_PATH = "/kaggle/input/datasets/kmkimmy/thai-depression/Thai Depression/valid.json"
TEST_PATH = "/kaggle/input/datasets/kmkimmy/thai-depression/Thai Depression/test.json"

def load_json_dataset(path):

    with open(path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    texts = []
    label_vectors = []

    for item in raw_data:

        if len(item) != 2:
            continue

        text, label = item

        text = str(text)

        if len(text.strip()) < 2:
            continue

        if label == "depression":
            label_vector = [0, 1]

        elif label == "no_depression":
            label_vector = [1, 0]

        else:
            continue

        texts.append(text)
        label_vectors.append(label_vector)

    df = pd.DataFrame({
        "Tweets": texts,
        "no_depression": [x[0] for x in label_vectors],
        "depression": [x[1] for x in label_vectors]
    })

    return df

train_df = load_json_dataset(TRAIN_PATH)
val_df = load_json_dataset(VALID_PATH)
test_df = load_json_dataset(TEST_PATH)

print("TRAIN:", len(train_df))
print("VALID:", len(val_df))
print("TEST :", len(test_df))

print("\nTRAIN LABEL COUNTS")
print(train_df[labels].sum())


TRAIN: 25075
VALID: 3344
TEST : 5015

TRAIN LABEL COUNTS
no_depression    12239
depression       12836
dtype: int64


In [9]:
def clean_text(text):

    text = str(text)

    text = text.replace("---- “ ----", " ")

    text = text.replace("\n", " ")

    text = re.sub(r"\s+", " ", text)

    return text.strip()

train_df['text'] = train_df['Tweets'].apply(clean_text)

val_df['text'] = val_df['Tweets'].apply(clean_text)

test_df['text'] = test_df['Tweets'].apply(clean_text)


In [10]:
# Dataset already split
print("TRAIN:", len(train_df))
print("VALID:", len(val_df))
print("TEST :", len(test_df))


TRAIN: 25075
VALID: 3344
TEST : 5015


In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

In [12]:
class EmotionDataset(Dataset):

    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.labels = df[labels].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

train_loader = DataLoader(EmotionDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(EmotionDataset(val_df), batch_size=BATCH_SIZE)
test_loader  = DataLoader(EmotionDataset(test_df), batch_size=BATCH_SIZE)

In [13]:
class Model(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

        self.decoder = Decoder(
            dim=DECODER_DIM,
            depth=DECODER_DEPTH,
            heads=DECODER_HEADS
        )

        self.fc = nn.Linear(DECODER_DIM, NUM_CLASSES)

    def forward(self, input_ids, attention_mask):

        x = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        x = self.decoder(
            x,
            mask=attention_mask.bool()
        )

        # mean pooling
        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(1) / mask.sum(1)

        return self.fc(x)

model = Model().to(DEVICE)

# ✅ แยก LR: encoder ต่ำ, decoder/fc สูง
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 2e-5},
    {"params": model.decoder.parameters(), "lr": 1e-3},
    {"params": model.fc.parameters(), "lr": 1e-3},
], weight_decay=0.01)

criterion = nn.BCEWithLogitsLoss()

# ✅ Warm-up scheduler
from transformers import get_linear_schedule_with_warmup

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)


2026-05-20 03:56:51.960779: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779249412.155949      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779249412.204805      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779249412.646046      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779249412.646088      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779249412.646090      23 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

In [14]:
@torch.no_grad()
def evaluate(loader):
    
    model.eval()
    
    preds = []
    
    targets = []
    
    for input_ids, attention_mask, y in loader:
        
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        
        logits = model(input_ids, attention_mask)
        
        probs = torch.sigmoid(logits)
        
        # ❗ baseline threshold = 0.5
        pred = (probs > 0.5).int().cpu().numpy()
        preds.append(pred)
        
        targets.append(y.numpy())
        
    y_pred = np.vstack(preds)
    y_true = np.vstack(targets)
    
    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    
    return f1_micro, f1_macro, acc

In [15]:
import os

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SAVE_EVERY_N_EPOCHS = 10
RESUME_CHECKPOINT_PATH = "/kaggle/input/datasets/kmkimmy/last-checkpoint/last_checkpoint.pt"

start_epoch = 0
best_f1_macro = 0.0

if RESUME_CHECKPOINT_PATH and os.path.exists(RESUME_CHECKPOINT_PATH):
    checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = checkpoint["epoch"]
    best_f1_macro = checkpoint.get("best_f1_macro", 0.0)
    print(f"Resumed from epoch {start_epoch}, best F1 macro: {best_f1_macro:.4f}")
else:
    print("Training from scratch")


Resumed from epoch 48, best F1 macro: 0.8350


In [16]:
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0

    for input_ids, attention_mask, y in train_loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        y = y.float().to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    f1_micro, f1_macro, acc = evaluate(val_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"loss={total_loss:.4f} | "
        f"f1_micro={f1_micro:.4f} | "
        f"f1_macro={f1_macro:.4f} | "
        f"acc={acc:.4f}"
    )

    # Best model checkpoint
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_f1_macro": best_f1_macro,
            "f1_micro": f1_micro,
            "accuracy": acc,
            "loss": total_loss,
        }, os.path.join(CHECKPOINT_DIR, "best_model.pt"))
        print(f"  >> Best model saved! (F1 macro: {best_f1_macro:.4f})")

    # Periodic checkpoint
    if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_f1_macro": best_f1_macro,
        }, os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.pt"))
        print(f"  >> Periodic checkpoint saved: epoch {epoch+1}")

# Save last checkpoint
torch.save({
    "epoch": epoch + 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "best_f1_macro": best_f1_macro,
}, os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt"))
print(f"\nLast checkpoint saved. Best F1 macro: {best_f1_macro:.4f}")


Epoch 49/100 | loss=4.1534 | f1_micro=0.8331 | f1_macro=0.8330 | acc=0.8331
Epoch 50/100 | loss=2.7966 | f1_micro=0.8389 | f1_macro=0.8386 | acc=0.8388
  >> Best model saved! (F1 macro: 0.8386)
  >> Periodic checkpoint saved: epoch 50
Epoch 51/100 | loss=3.2278 | f1_micro=0.8325 | f1_macro=0.8324 | acc=0.8325
Epoch 52/100 | loss=3.9658 | f1_micro=0.8334 | f1_macro=0.8334 | acc=0.8334
Epoch 53/100 | loss=3.0466 | f1_micro=0.8267 | f1_macro=0.8267 | acc=0.8266
Epoch 54/100 | loss=2.1318 | f1_micro=0.8304 | f1_macro=0.8303 | acc=0.8304
Epoch 55/100 | loss=2.7192 | f1_micro=0.8318 | f1_macro=0.8317 | acc=0.8316
Epoch 56/100 | loss=3.0980 | f1_micro=0.8313 | f1_macro=0.8313 | acc=0.8313
Epoch 57/100 | loss=2.5949 | f1_micro=0.8362 | f1_macro=0.8362 | acc=0.8361
Epoch 58/100 | loss=2.5709 | f1_micro=0.8382 | f1_macro=0.8381 | acc=0.8382
Epoch 59/100 | loss=2.9919 | f1_micro=0.8349 | f1_macro=0.8347 | acc=0.8349
Epoch 60/100 | loss=2.5503 | f1_micro=0.8331 | f1_macro=0.8331 | acc=0.8331
  >> 

In [17]:
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(best_ckpt_path):
    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_ckpt["model_state_dict"])
    print(f"Loaded best model from epoch {best_ckpt['epoch']}, F1 macro: {best_ckpt['best_f1_macro']:.4f}")


Loaded best model from epoch 87, F1 macro: 0.8389


In [18]:
f1_micro, f1_macro, acc = evaluate(test_loader)

print("\nFINAL TEST")
print("F1 Micro :", f1_micro)
print("F1 Macro :", f1_macro)
print("Accuracy :", acc)

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():

    for input_ids, attention_mask, y in test_loader:

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_ids, attention_mask)

        preds = (torch.sigmoid(logits) > 0.5).int()

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        all_targets,
        all_preds,
        target_names=labels,
        zero_division=0
    )
)

print("\nCONFUSION MATRIX")

true_labels = np.argmax(all_targets, axis=1)
pred_labels = np.argmax(all_preds, axis=1)

cm = confusion_matrix(
    true_labels,
    pred_labels
)

print(cm)



FINAL TEST
F1 Micro : 0.8279162512462612
F1 Macro : 0.8276740789653751
Accuracy : 0.8279162512462612

CLASSIFICATION REPORT
               precision    recall  f1-score   support

no_depression       0.83      0.81      0.82      2448
   depression       0.82      0.85      0.83      2567

    micro avg       0.83      0.83      0.83      5015
    macro avg       0.83      0.83      0.83      5015
 weighted avg       0.83      0.83      0.83      5015
  samples avg       0.83      0.83      0.83      5015


CONFUSION MATRIX
[[1982  466]
 [ 397 2170]]


In [19]:
# ✅ บันทึกโมเดล
torch.save(model.state_dict(), "wangchanberta_xtransformer_sied.pt")
print("Model saved to wangchanberta_xtransformer_sied.pt")


Model saved to wangchanberta_xtransformer_sied.pt


In [20]:
# ✅ โหลดโมเดลจาก best checkpoint
loaded_model = Model().to(DEVICE)

best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
    loaded_model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded best model from epoch {ckpt['epoch']}, F1 macro: {ckpt['best_f1_macro']:.4f}")
else:
    loaded_model.load_state_dict(
        torch.load("wangchanberta_xtransformer_sied.pt", map_location=DEVICE, weights_only=False)
    )
    print("Loaded model from wangchanberta_xtransformer_sied.pt")

loaded_model.eval()


Loaded best model from epoch 87, F1 macro: 0.8389


Model(
  (encoder): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(25005, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): CamembertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNo

In [21]:
from sklearn.metrics import classification_report, precision_score, recall_score

@torch.no_grad()
def predict_and_report(model, loader):
    model.eval()
    all_preds = []
    all_targets = []

    for input_ids, attention_mask, y in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits)
        pred = (probs > 0.5).int().cpu().numpy()

        all_preds.append(pred)
        all_targets.append(y.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    # ✅ Per-label Precision / Recall / F1-score
    print("=" * 60)
    print("Per-Label Classification Report")
    print("=" * 60)
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=labels,
            digits=4,
            zero_division=0
        )
    )

    # ✅ Overall metrics
    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    print("=" * 60)
    print(f"F1 Micro  : {f1_micro:.4f}")
    print(f"F1 Macro  : {f1_macro:.4f}")
    print(f"Accuracy  : {acc:.4f}")
    print("=" * 60)

# เรียกใช้กับ loaded model
predict_and_report(loaded_model, test_loader)


Per-Label Classification Report
               precision    recall  f1-score   support

no_depression     0.8331    0.8096    0.8212      2448
   depression     0.8232    0.8453    0.8341      2567

    micro avg     0.8279    0.8279    0.8279      5015
    macro avg     0.8282    0.8275    0.8277      5015
 weighted avg     0.8281    0.8279    0.8278      5015
  samples avg     0.8279    0.8279    0.8279      5015

F1 Micro  : 0.8279
F1 Macro  : 0.8277
Accuracy  : 0.8279
